# Unitree Go1: Joystick Task Training
We use Mujoco Playground to train the Go1 for Joystick tasks on Flat Ground for now. For this purpose, we use the brax based PPO implementation from the Model-based policy optimizers repository, since we can use this implementation for TaCoS based agents as well.

## Prerequisites

We check if the installation was successful and optimize settings for the GPU

In [1]:
import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

Tue May  6 09:46:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:C1:00.0 Off |                  Off |
| 31%   35C    P5             48W /  450W |       2MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

We prepare some packages for rendering the simulation to a video

In [2]:
import json
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

We import the imports from brax, playground, mjx, and setup our PPO Implementation from the mbpo repository

In [1]:
from datetime import datetime
import functools
import os

In [2]:
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
from jax import numpy as jp
from matplotlib import pyplot as plt
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

In [3]:
from typing import Any, Dict, Sequence, Tuple, Union

In [4]:
# brax imports
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState

In [5]:
# Import PPO from wtc (adapted version with privileged state)
from wtc.agents.ppo.ppo_brax_env import PPO

In [8]:
#Check available devices
devices = jax.devices()
print("Available devices:", devices)

# Check default backend
print("Default backend:", jax.default_backend())
!nvidia-smi


Available devices: [CudaDevice(id=0)]
Default backend: gpu
Tue May  6 09:47:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:C1:00.0 Off |                  Off |
| 32%   36C    P2             27W /  450W |   18556MiB /  24564MiB |      0%      Default |
|                                         |                        |             

/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [6]:
from mujoco_playground import wrapper
from mujoco_playground import registry

Check whether playground was succesfully installed by listing all available envs in playground

In [7]:
registry.locomotion.ALL_ENVS

('BarkourJoystick',
 'BerkeleyHumanoidJoystickFlatTerrain',
 'BerkeleyHumanoidJoystickRoughTerrain',
 'G1JoystickFlatTerrain',
 'G1JoystickRoughTerrain',
 'Go1JoystickFlatTerrain',
 'Go1JoystickRoughTerrain',
 'Go1Getup',
 'Go1Handstand',
 'Go1Footstand',
 'H1InplaceGaitTracking',
 'H1JoystickGaitTracking',
 'Op3Joystick',
 'SpotFlatTerrainJoystick',
 'SpotGetup',
 'SpotJoystickGaitTracking',
 'T1JoystickFlatTerrain',
 'T1JoystickRoughTerrain')


## Training with PPO

In [8]:
env_name = 'Go1JoystickFlatTerrain'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)

In [9]:
env_cfg

Kd: 0.5
Kp: 35.0
action_repeat: 1
action_scale: 0.5
command_config:
  a:
  - 1.5
  - 0.8
  - 1.2
  b:
  - 0.9
  - 0.25
  - 0.5
ctrl_dt: 0.02
episode_length: 1000
history_len: 1
noise_config:
  level: 1.0
  scales:
    gravity: 0.05
    gyro: 0.2
    joint_pos: 0.03
    joint_vel: 1.5
    linvel: 0.1
pert_config:
  enable: false
  kick_durations:
  - 0.05
  - 0.2
  kick_wait_times:
  - 1.0
  - 3.0
  velocity_kick:
  - 0.0
  - 3.0
reward_config:
  max_foot_height: 0.1
  scales:
    action_rate: -0.01
    ang_vel_xy: -0.05
    dof_pos_limits: -1.0
    energy: -0.001
    feet_air_time: 0.1
    feet_clearance: -2.0
    feet_height: -0.2
    feet_slip: -0.1
    lin_vel_z: -0.5
    orientation: -5.0
    pose: 0.5
    stand_still: -1.0
    termination: -1.0
    torques: -0.0002
    tracking_ang_vel: 0.5
    tracking_lin_vel: 1.0
  tracking_sigma: 0.25
sim_dt: 0.004
soft_joint_pos_limit_factor: 0.95

In [10]:
env.dt

0.02

We now get the policy parameters of the Go1 used in the playground. These parameters are tested and perform good in sim to real transfer.

In [11]:
from mujoco_playground.config import locomotion_params
ppo_params = locomotion_params.brax_ppo_config(env_name)

In [12]:
ppo_params

action_repeat: 1
batch_size: 256
discounting: 0.97
entropy_cost: 0.01
episode_length: 1000
learning_rate: 0.0003
max_grad_norm: 1.0
network_factory:
  policy_hidden_layer_sizes: &id001 !!python/tuple
  - 512
  - 256
  - 128
  policy_obs_key: state
  value_hidden_layer_sizes: *id001
  value_obs_key: privileged_state
normalize_observations: true
num_envs: 8192
num_evals: 10
num_minibatches: 32
num_resets_per_eval: 1
num_timesteps: 200000000
num_updates_per_batch: 4
reward_scaling: 1.0
unroll_length: 20

Using this params, we can now setup the optimizer. However, first we need the domain randomization function. Domain randomization helps us bridge the sim2real gap, by randomizing the environmnet, such that the real environment resembles another variation of the simulation environment. For this, we can simply use the function from the playground

The domain randomization function from the playground samples the friction coefficient of the ground, the internal joint resistance, armature, shifts the torso's center of mass by a sampled scalar, alters the body masses, a sampled increase of weight and different starting joint positions.

In [13]:
import jax
from functools import partial
ppo_config = dict(ppo_params)

In [14]:
randomization_fn = registry.get_domain_randomizer(env_name)

In [15]:
import inspect
inspect.signature(randomization_fn)

<Signature (model: mujoco.mjx._src.types.Model, rng: jax.Array)>

We can now setup the optimizer with this domain randomized environment

In [ ]:
from jax.nn import swish
policy_hidden_layer_sizes = list(ppo_config['network_factory']['policy_hidden_layer_sizes'])
critic_hidden_layer_sizes = list(ppo_config['network_factory']['value_hidden_layer_sizes'])
optimizer = PPO(
            environment=env,
            num_timesteps=ppo_config['num_timesteps'],
            episode_length=ppo_config['episode_length'],
            action_repeat=ppo_config['action_repeat'],
            num_envs=ppo_config['num_envs'],
            lr=ppo_config['learning_rate'],
            wd=0.,
            entropy_cost=ppo_config['entropy_cost'],
            unroll_length=ppo_config['unroll_length'],
            discounting=ppo_config['discounting'],
            batch_size=ppo_config['batch_size'],
            num_minibatches=ppo_config['num_minibatches'],
            num_updates_per_batch=ppo_config['num_updates_per_batch'],
            num_evals=ppo_config['num_evals'],
            normalize_observations=ppo_config['normalize_observations'],
            reward_scaling=ppo_config['reward_scaling'],
            max_grad_norm=ppo_config['max_grad_norm'],
            clipping_epsilon=0.3,
            gae_lambda=0.95,
            policy_hidden_layer_sizes=policy_hidden_layer_sizes,
            policy_activation=swish,
            critic_hidden_layer_sizes=critic_hidden_layer_sizes,
            critic_activation=swish,
            deterministic_eval=False,
            normalize_advantage=True,
            wandb_logging=False,
            policy_obs_key = ppo_config['network_factory']['policy_obs_key'],
            value_obs_key = ppo_config['network_factory']['value_obs_key'],
        )

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

We start training using our optimizer (not working yet)

In [ ]:
rng = jax.random.PRNGKey(0)
init_state = env.reset(rng)

In [ ]:
init_state.obs

In [ ]:
print('Before Inference')
policy_params, metrics = optimizer.run_training(key=jax.random.PRNGKey(0), progress_fn=progress)
print('After Inference')

In [ ]:
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

We tryout the brax ppo optimizer

In [ ]:
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
randomizer = registry.get_domain_randomizer(env_name)
ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    randomization_fn=randomizer,
    progress_fn=progress
)
make_inference_fn, params, metrics = train_fn(
    environment=env,
    eval_env=registry.load(env_name, config=env_cfg),
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

We save the policy now and perform the rollouts. We save the frames and generate the video locally.

In [ ]:
import pickle
# Create a directory to store the outputs if it doesn't exist
os.makedirs("trained_policy_mbpo_dr", exist_ok=True)

# Save the trained parameters
with open("trained_policy_mbpo_dr/mbpo_dr_ppo_playground.pkl", "wb") as f:
    pickle.dump(policy_params, f)

# Optionally, save the metrics too (e.g., episode returns, losses, etc.)
with open("trained_policy_mbpo_dr/mbpo_dr_policy_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)

# Save the environment config if you want to reproduce this setup locally
with open("trained_policy_mbpo_dr/env_config.pkl", "wb") as f:
    pickle.dump(env_cfg, f)

print("✅ Policy and metadata saved to 'trained_policy_mbpo_dr/'")

In [ ]:
import cloudpickle
with open("Apr25_fixedRewards/PPO_fixSetup/Policies/policy_params_vo3wweqf.pkl", "rb") as f:
    policy_params = cloudpickle.load(f)

In [ ]:
# Enable perturbation in the eval env.
env_cfg = registry.get_default_config(env_name)
env_cfg.pert_config.enable = False
env_cfg.pert_config.velocity_kick = [3.0, 6.0]
env_cfg.pert_config.kick_wait_times = [5.0, 15.0]
env_cfg.command_config.a = [1.5, 0.8, 2*jp.pi]
eval_env = registry.load(env_name, config=env_cfg)
velocity_kick_range = [0.0, 0.0]  # Disable velocity kick.
kick_duration_range = [0.05, 0.2]

jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)
jit_inference_fn = jax.jit(optimizer.make_policy(policy_params, deterministic=True))

In [ ]:
#@title Rollout and Render
from mujoco_playground._src.gait import draw_joystick_command

def generate_command(t):
    """Smoothly varying command over time t (steps)."""
    x_vel = -0.25 + 0.0005 * t  # Increase x linearly
    y_vel = 0.2 * jp.sin(2 * jp.pi * 0.001 * t)  # small oscillation in y
    yaw_vel = 0.0  # no rotation
    return jp.array([x_vel, y_vel, yaw_vel])

In [ ]:

rng = jax.random.PRNGKey(0)
rollout = []
modify_scene_fns = []

swing_peak = []
rewards = []
linvel = []
angvel = []
track = []
foot_vel = []
rews = []
contact = []
num_actions = 0
x_vel = 0
y_vel = 0
yaw_vel = 3.14
command = jp.array([x_vel, y_vel, yaw_vel])

state = jit_reset(rng)
state.info["command"] = command
total_reward = 0
while num_actions < env_cfg.episode_length:
  act_rng, rng = jax.random.split(rng)
  ctrl, _ = jit_inference_fn(state.obs, act_rng)
  state = jit_step(state, ctrl)
  num_actions += 1
  state.info["command"] = command
  rews.append(
      {k: v for k, v in state.metrics.items() if k.startswith("reward/")}
  )
  rollout.append(state)
  swing_peak.append(state.info["swing_peak"])
  rewards.append(
      {k[7:]: v for k, v in state.metrics.items() if k.startswith("reward/")}
  )
  total_reward += state.reward
  linvel.append(env.get_global_linvel(state.data))
  angvel.append(env.get_gyro(state.data))
  track.append(
      env._reward_tracking_lin_vel(
          state.info["command"], env.get_local_linvel(state.data)
      )
  )

  feet_vel = state.data.sensordata[env._foot_linvel_sensor_adr]
  vel_xy = feet_vel[..., :2]
  vel_norm = jp.sqrt(jp.linalg.norm(vel_xy, axis=-1))
  foot_vel.append(vel_norm)

  contact.append(state.info["last_contact"])

  xyz = np.array(state.data.xpos[env._torso_body_id])
  xyz += np.array([0, 0, 0.2])
  x_axis = state.data.xmat[env._torso_body_id, 0]
  yaw = -np.arctan2(x_axis[1], x_axis[0])
  modify_scene_fns.append(
      functools.partial(
          draw_joystick_command,
          cmd=state.info["command"],
          xyz=xyz,
          theta=yaw,
          scl=abs(state.info["command"][0])
          / env_cfg.command_config.a[0],
      )
  )
print(f"Agent took {num_actions} actions")
print(f"Agent got {total_reward} reward")
render_every = 2
fps = 1.0 / eval_env.dt / render_every
traj = rollout[::render_every]
mod_fns = modify_scene_fns[::render_every]

scene_option = mujoco.MjvOption()
scene_option.geomgroup[2] = True
scene_option.geomgroup[3] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = True

frames = eval_env.render(
    traj,
    camera="track",
    scene_option=scene_option,
    width=640,
    height=480,
    modify_scene_fns=mod_fns,
)
os.makedirs("frames_ppo_fixObs", exist_ok=True)
for i, frame in enumerate(frames):
    plt.imsave(f"frames_ppo_fixObs/frame_{i:04d}.png", frame)

We visualize the feet positions and the positional drift compared to the commanded linear and angular velocity

In [ ]:
swing_peak = jp.array(swing_peak)
names = ["FR", "FL", "RR", "RL"]
colors = ["r", "g", "b", "y"]
fig, axs = plt.subplots(2, 2)
for i, ax in enumerate(axs.flat):
  ax.plot(swing_peak[:, i], color=colors[i])
  ax.set_ylim([0, env_cfg.reward_config.max_foot_height * 1.25])
  ax.axhline(env_cfg.reward_config.max_foot_height, color="k", linestyle="--")
  ax.set_title(names[i])
  ax.set_xlabel("time")
  ax.set_ylabel("height")
plt.tight_layout()
plt.show()

linvel_x = jp.array(linvel)[:, 0]
linvel_y = jp.array(linvel)[:, 1]
angvel_yaw = jp.array(angvel)[:, 2]

# Plot whether velocity is within the command range.
linvel_x = jp.convolve(linvel_x, jp.ones(10) / 10, mode="same")
linvel_y = jp.convolve(linvel_y, jp.ones(10) / 10, mode="same")
angvel_yaw = jp.convolve(angvel_yaw, jp.ones(10) / 10, mode="same")

fig, axes = plt.subplots(3, 1, figsize=(10, 10))
axes[0].plot(linvel_x)
axes[1].plot(linvel_y)
axes[2].plot(angvel_yaw)

axes[0].set_ylim(
    -env_cfg.command_config.a[0], env_cfg.command_config.a[0]
)
axes[1].set_ylim(
    -env_cfg.command_config.a[1], env_cfg.command_config.a[1]
)
axes[2].set_ylim(
    -env_cfg.command_config.a[2], env_cfg.command_config.a[2]
)

for i, ax in enumerate(axes):
  ax.axhline(state.info["command"][i], color="red", linestyle="--")

labels = ["dx", "dy", "dyaw"]
for i, ax in enumerate(axes):
  ax.set_ylabel(labels[i])

## Training using TaCoS

We now use the Switch Cost Wrapper for TaCoS with a switch cost of 0.1 to wrap the Go1 environment with the time-adaptive agent. Our aim is to use this extension of the Go1 MDP and minimize the number of interactions the Go1 has while executing various tasks. We will use the TaCoS framework on top of PPO.

In [ ]:
from wtc.wrappers.ih_switching_cost_mjx import ConstantSwitchCost, IHSwitchCostWrapper

For now, we allow the quadruped to take an action for 5 timesteps at maximum, and 1 at minimum. Here, the dt we need to pass is the control dt from the Go1 configurations, since the simulation dt is used to evaluate the state, whereas the control dt is the dt at which actions are sent to the controller. 

In [16]:
env_name = 'Go1JoystickFlatTerrain'
env = registry.load(env_name)
env_cfg = dict(registry.get_default_config(env_name))

In [17]:
env_cfg

{'Kd': 0.5,
 'Kp': 35.0,
 'action_repeat': 1,
 'action_scale': 0.5,
 'command_config': a:
 - 1.5
 - 0.8
 - 1.2
 b:
 - 0.9
 - 0.25
 - 0.5,
 'ctrl_dt': 0.02,
 'episode_length': 1000,
 'history_len': 1,
 'noise_config': level: 1.0
 scales:
   gravity: 0.05
   gyro: 0.2
   joint_pos: 0.03
   joint_vel: 1.5
   linvel: 0.1,
 'pert_config': enable: false
 kick_durations:
 - 0.05
 - 0.2
 kick_wait_times:
 - 1.0
 - 3.0
 velocity_kick:
 - 0.0
 - 3.0,
 'reward_config': max_foot_height: 0.1
 scales:
   action_rate: -0.01
   ang_vel_xy: -0.05
   dof_pos_limits: -1.0
   energy: -0.001
   feet_air_time: 0.1
   feet_clearance: -2.0
   feet_height: -0.2
   feet_slip: -0.1
   lin_vel_z: -0.5
   orientation: -5.0
   pose: 0.5
   stand_still: -1.0
   termination: -1.0
   torques: -0.0002
   tracking_ang_vel: 0.5
   tracking_lin_vel: 1.0
 tracking_sigma: 0.25,
 'sim_dt': 0.004,
 'soft_joint_pos_limit_factor': 0.95}

In [18]:
env.action_size

12

In [19]:
ppo_config = dict(locomotion_params.brax_ppo_config(env_name))

In [20]:
ppo_config

{'action_repeat': 1,
 'batch_size': 256,
 'discounting': 0.97,
 'entropy_cost': 0.01,
 'episode_length': 1000,
 'learning_rate': 0.0003,
 'max_grad_norm': 1.0,
 'network_factory': policy_hidden_layer_sizes: &id001 !!python/tuple
 - 512
 - 256
 - 128
 policy_obs_key: state
 value_hidden_layer_sizes: *id001
 value_obs_key: privileged_state,
 'normalize_observations': True,
 'num_envs': 8192,
 'num_evals': 10,
 'num_minibatches': 32,
 'num_resets_per_eval': 1,
 'num_timesteps': 200000000,
 'num_updates_per_batch': 4,
 'reward_scaling': 1.0,
 'unroll_length': 20}

In [ ]:
import jax.numpy as jnp
tmax = 3
tmin = 1
switch_cost_value = 0
tacos_env = IHSwitchCostWrapper(env=env, episode_steps=ppo_config['episode_length'], 
                                min_time_between_switches=tmin, max_time_between_switches=tmax, switch_cost = ConstantSwitchCost(value=jnp.array(switch_cost_value)),
                                discounting = ppo_config['discounting'], time_as_part_of_state=True, sim_dt = env_cfg['sim_dt'])

In [ ]:
env.observation_size['state'][0]+1

In [ ]:
obs_size = {
    k: (v[0]+1, ) for k,v in env.observation_size.items()
}

In [ ]:
env.action_space

We setup the optimizer for TaCoS PPO

In [ ]:
from jax.nn import swish
from wtc.utils import discrete_to_continuous_discounting
policy_hidden_layer_sizes = list(ppo_config['network_factory']['policy_hidden_layer_sizes'])
critic_hidden_layer_sizes = list(ppo_config['network_factory']['value_hidden_layer_sizes'])
continuous_discounting = discrete_to_continuous_discounting(discrete_discounting=ppo_config['discounting'],
                                                            dt=env_cfg['ctrl_dt'])
optimizer = PPO(
            environment=tacos_env,
            num_timesteps=ppo_config['num_timesteps'],
            episode_length=ppo_config['episode_length'],
            action_repeat=ppo_config['action_repeat'],
            num_envs=ppo_config['num_envs'],
            lr=ppo_config['learning_rate'],
            wd=0.,
            entropy_cost=ppo_config['entropy_cost'],
            unroll_length=ppo_config['unroll_length'],
            discounting=ppo_config['discounting'],
            batch_size=ppo_config['batch_size'],
            num_minibatches=ppo_config['num_minibatches'],
            num_updates_per_batch=ppo_config['num_updates_per_batch'],
            num_evals=ppo_config['num_evals'],
            normalize_observations=ppo_config['normalize_observations'],
            reward_scaling=ppo_config['reward_scaling'],
            max_grad_norm=ppo_config['max_grad_norm'],
            clipping_epsilon=0.3,
            gae_lambda=0.95,
            policy_hidden_layer_sizes=policy_hidden_layer_sizes,
            policy_activation=swish,
            critic_hidden_layer_sizes=critic_hidden_layer_sizes,
            critic_activation=swish,
            deterministic_eval=True,
            normalize_advantage=True,
            wandb_logging=False,
            policy_obs_key = ppo_config['network_factory']['policy_obs_key'],
            value_obs_key = ppo_config['network_factory']['value_obs_key'],
            return_best_model = True,
            min_time_between_switches = tmin,
            max_time_between_switches = tmax,
            env_dt = env_cfg['ctrl_dt'],
            randomization_fn = randomization_fn,
            seed = 4,
        )

We try out the brax ppo optimizer

In [21]:
xdata, ydata, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
    clear_output(wait=True)
    times.append(datetime.now())
    xdata.append(num_steps)
    ydata.append(metrics['eval/episode_reward'])
    y_dataerr.append(metrics['eval/episode_reward_std'])
    plt.xlim([0, ppo_params['num_timesteps'] * 1.25])
    plt.xlabel('# environment steps')
    plt.ylabel('reward per episode')
    plt.title(f"y={ydata[-1]:.3f}")
    plt.errorbar(xdata, ydata, yerr=y_dataerr, color="blue")
    display(plt.gcf())



In [22]:
from wtc.wrappers.ih_switching_cost_mjx import wrap

In [23]:
switch_cost_wrap = functools.partial(wrap, switch_cost_wrapper=True, 
                                 switch_cost=0, min_time_between_switches = 1, max_time_between_switches = 3, discounting=ppo_config['discounting'],
                                )

In [25]:
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
randomizer = registry.get_domain_randomizer(env_name)
ppo_training_params = dict(ppo_params)
env_cfg = registry.get_default_config(env_name)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    randomization_fn=randomizer,
    progress_fn=progress
)
make_inference_fn, params, metrics = train_fn(
    environment=env,
    eval_env=registry.load(env_name, config=env_cfg),
    wrap_env_fn=switch_cost_wrap,
)
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

UnexpectedTracerError: Encountered an unexpected tracer. A function transformed by JAX had a side effect, allowing for a reference to an intermediate value with type float32[51,3] wrapped in a BatchTracer to escape the scope of the transformation.
JAX transformations require that functions explicitly return their outputs, and disallow saving intermediate values to global state.
To catch the leak earlier, try setting the environment variable JAX_CHECK_TRACER_LEAKS or using the `jax.checking_leaks` context manager.
See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.UnexpectedTracerError

We rollout the policy now and save it

In [ ]:
import pickle
# Create a directory to store the outputs if it doesn't exist
os.makedirs("trained_policy_tacosppo", exist_ok=True)

# Save the trained parameters
with open("trained_policy_tacosppo/tacosppo_playground.pkl", "wb") as f:
    pickle.dump(policy_params, f)

# Optionally, save the metrics too (e.g., episode returns, losses, etc.)
with open("trained_policy_tacosppo/tacosppo_policy_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)

# Save the environment config if you want to reproduce this setup locally
with open("trained_policy_tacosppo/env_config.pkl", "wb") as f:
    pickle.dump(env_cfg, f)

print("✅ Policy and metadata saved to 'trained_policy_tacosppo/'")

We load up: PPO-Tacos with s in [1,5], cost c = 0.01, seed = 3

In [ ]:
import cloudpickle
with open("Apr25_fixedRewards/TacosRewardFix/Policies/policy_params_jausn7tc.pkl", "rb") as f:
    policy_params = cloudpickle.load(f)

In [ ]:
# Enable perturbation in the eval env.
env_cfg = registry.get_default_config(env_name)
env_cfg.pert_config.enable = False
env_cfg.pert_config.velocity_kick = [3.0, 6.0]
env_cfg.pert_config.kick_wait_times = [5.0, 15.0]
env_cfg.command_config.a = [1.5, 0.8, 2*jp.pi]
eval_env = registry.load(env_name, config=env_cfg)
eval_env = IHSwitchCostWrapper(env=eval_env, episode_steps=ppo_config['episode_length'], 
                                min_time_between_switches=1, max_time_between_switches=3, switch_cost = ConstantSwitchCost(value=jnp.array(0.0)),
                                discounting = ppo_config['discounting'], time_as_part_of_state=True)
velocity_kick_range = [0.0, 0.0]  # Disable velocity kick.
kick_duration_range = [0.05, 0.2]

jit_reset = jax.jit(eval_env.reset)
jit_step = eval_env.simulation_step # we just don't jit it
jit_inference_fn = jax.jit(optimizer.make_policy(policy_params, deterministic=True))

In [ ]:
env_cfg.reward_config

In [ ]:
#@title Rollout and Render
from mujoco_playground._src.gait import draw_joystick_command

x_vel = 0.0  #@param {type: "number"}
y_vel = 0.0  #@param {type: "number"}
yaw_vel = 3.14  #@param {type: "number"}


rng = jax.random.PRNGKey(0)
rollout = []
render_rollout = []
modify_scene_fns = []

swing_peak = []
rewards = []
linvel = []
angvel = []
track = []
foot_vel = []
rews = []
contact = []
command = jp.array([x_vel, y_vel, yaw_vel])
num_steps = 0
env_steps = 0
time_predictions = []

state = jit_reset(rng)
state.info["command"] = command
total_reward = state.reward
while env_steps < ppo_config['episode_length']:
  act_rng, rng = jax.random.split(rng)
  ctrl, _ = jit_inference_fn(state.obs, act_rng)
  num_steps += 1
  env_steps += eval_env.compute_time(pseudo_time=ctrl[-1], t_upper = 3, t_lower=1)
  time_predictions.append(eval_env.compute_time(pseudo_time=ctrl[-1], t_upper=3, t_lower=1))
  state, inner = jit_step(state, ctrl)
  state.info["command"] = command
  rews.append(
      {k: v for k, v in state.metrics.items() if k.startswith("reward/")}
  )
  total_reward += state.reward
  rollout.append(state)
  render_rollout.append(inner)
  swing_peak.append(state.info["swing_peak"])
  current_reward = {k[7:]: v for k, v in state.metrics.items() if k.startswith("reward/")}
  rewards.append(current_reward)
  linvel.append(tacos_env.env.get_global_linvel(state.data))
  angvel.append(tacos_env.env.get_gyro(state.data))
  track.append(
      tacos_env.env._reward_tracking_lin_vel(
          state.info["command"], tacos_env.env.get_local_linvel(state.data)
      )
  )

  feet_vel = state.data.sensordata[tacos_env.env._foot_linvel_sensor_adr]
  vel_xy = feet_vel[..., :2]
  vel_norm = jp.sqrt(jp.linalg.norm(vel_xy, axis=-1))
  foot_vel.append(vel_norm)

  contact.append(state.info["last_contact"])

  xyz = np.array(state.data.xpos[tacos_env.env._torso_body_id])
  xyz += np.array([0, 0, 0.2])
  x_axis = state.data.xmat[tacos_env.env._torso_body_id, 0]
  yaw = -np.arctan2(x_axis[1], x_axis[0])
  modify_scene_fns.append(
      functools.partial(
          draw_joystick_command,
          cmd=state.info["command"],
          xyz=xyz,
          theta=yaw,
          scl=abs(state.info["command"][0])
          / env_cfg.command_config.a[0],
      )
  )
print(f"The total reward is {total_reward}")
print(f"The agent took {num_steps} actions")

In [ ]:
# How many steps in each inner list
import jax.tree_util as jtu
render_rollout = [part[0] if isinstance(part, tuple) else part for part in render_rollout]

# Now safely get inner lengths
inner_lengths = [jtu.tree_leaves(part)[0].shape[0] for part in render_rollout]

# Flatten all trees properly
flat_render_states = jtu.tree_map(
    lambda *xs: jnp.concatenate(xs, axis=0),
    *render_rollout
)

# Duplicate mod_fns accordingly
expanded_mod_fns = [
    fn for fn, count in zip(modify_scene_fns, inner_lengths)
    for _ in range(count)
]

render_every = 2  # You can set this higher if rendering takes too long
fps = 1.0 / eval_env.dt / render_every
traj = jtu.tree_map(lambda x: x[::render_every], flat_render_states)
num_render_frames = jtu.tree_leaves(traj)[0].shape[0]

# Re-split the PyTree back into a list of states
traj_split = [
    jtu.tree_map(lambda x: x[i], traj)
    for i in range(num_render_frames)
]
mod_fns = expanded_mod_fns[::render_every]

scene_option = mujoco.MjvOption()
scene_option.geomgroup[2] = True
scene_option.geomgroup[3] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = True

# Render the trajectory of all intermediate inner states
frames = eval_env.env.render(
    traj_split,
    camera="track",
    scene_option=scene_option,
    width=640,
    height=480,
    modify_scene_fns=mod_fns,
)

os.makedirs("frames_tacos4_correctObs", exist_ok=True)
for i, frame in enumerate(frames):
    plt.imsave(f"frames_tacos4_correctObs/frame_{i:04d}.png", frame)

In [ ]:
#@title Plot each foot in a 2x2 grid.

swing_peak = jp.array(swing_peak)
names = ["FR", "FL", "RR", "RL"]
colors = ["r", "g", "b", "y"]
fig, axs = plt.subplots(2, 2)
for i, ax in enumerate(axs.flat):
  ax.plot(swing_peak[:, i], color=colors[i])
  ax.set_ylim([0, env_cfg.reward_config.max_foot_height * 1.25])
  ax.axhline(env_cfg.reward_config.max_foot_height, color="k", linestyle="--")
  ax.set_title(names[i])
  ax.set_xlabel("time")
  ax.set_ylabel("height")
plt.tight_layout()
plt.show()

linvel_x = jp.array(linvel)[:, 0]
linvel_y = jp.array(linvel)[:, 1]
angvel_yaw = jp.array(angvel)[:, 2]

# Plot whether velocity is within the command range.
linvel_x = jp.convolve(linvel_x, jp.ones(10) / 10, mode="same")
linvel_y = jp.convolve(linvel_y, jp.ones(10) / 10, mode="same")
angvel_yaw = jp.convolve(angvel_yaw, jp.ones(10) / 10, mode="same")

fig, axes = plt.subplots(3, 1, figsize=(10, 10))
axes[0].plot(linvel_x)
axes[1].plot(linvel_y)
axes[2].plot(angvel_yaw)

axes[0].set_ylim(
    -env_cfg.command_config.a[0], env_cfg.command_config.a[0]
)
axes[1].set_ylim(
    -env_cfg.command_config.a[1], env_cfg.command_config.a[1]
)
axes[2].set_ylim(
    -env_cfg.command_config.a[2], env_cfg.command_config.a[2]
)

for i, ax in enumerate(axes):
  ax.axhline(state.info["command"][i], color="red", linestyle="--")

labels = ["dx", "dy", "dyaw"]
for i, ax in enumerate(axes):
  ax.set_ylabel(labels[i])